# 00 · Download and prepare the ECG cohort

Run this notebook from top to bottom in the project environment (`uv sync --locked`, then select `.venv/bin/python` as the kernel). It covers the Phase 1 workflow. Existing outputs are reused; choose a **new** output directory to repeat preparation with different settings. The source is [PhysioNet Apnea-ECG v1.0.0](https://physionet.org/content/apnea-ecg/1.0.0/).

In [1]:
import hashlib
import json
import math
import os
import platform
import shutil
import subprocess
import tempfile
import time
import tomllib
from datetime import UTC, datetime
from importlib.metadata import version
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from IPython.display import display
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import GroupShuffleSplit
from wfdb.processing import correct_peaks, gqrs_detect

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data/raw/apnea-ecg-1.0.0"
RUN = Path(os.environ.get("APNEA_RUN", str(ROOT / "data/processed/phase1")))
if not RUN.is_absolute():
    RUN = ROOT / RUN
CONFIG = ROOT / "configs/phase1.toml"
print("Raw:", RAW, "\nPrepared:", RUN, "\nConfig:", CONFIG)

Raw: <project-root>/data/raw/apnea-ecg-1.0.0 
Prepared: <project-root>/data/processed/phase1 
Config: <project-root>/configs/phase1.toml


## Implementation · Download and source verification

The following code is the complete implementation for this stage. Continue to the numbered workflow below to run it.

In [2]:
BASE_URL = "https://physionet.org/files/apnea-ecg/1.0.0/"
LEARNING_RECORDS = (
    [f"a{i:02}" for i in range(1, 21)]
    + [f"b{i:02}" for i in range(1, 6)]
    + [f"c{i:02}" for i in range(1, 11)]
)
METADATA = ("RECORDS", "ANNOTATORS", "additional-information.txt")


def sha256(path: Path) -> str:
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def validate_records(records: list[str]) -> list[str]:
    if not records or len(set(records)) != len(records) or set(records) - set(LEARNING_RECORDS):
        raise ValueError("Use unique original learning record IDs (a01–a20, b01–b05, c01–c10)")
    return sorted(records)


def fetch_verified(url: str, destination: Path, expected_hash: str) -> str:
    """Verify caches too; an interrupted/invalid transfer never replaces a raw file."""
    destination = Path(destination)
    if destination.is_file() and sha256(destination) == expected_hash:
        return expected_hash
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(
                dir=destination.parent, suffix=".part", delete=False
            ) as output:
                temporary = Path(output.name)
                with urlopen(url, timeout=60) as response:
                    shutil.copyfileobj(response, output)
            if sha256(temporary) != expected_hash:
                raise ValueError(f"SHA-256 checksum mismatch: {destination.name}")
            temporary.replace(destination)
            return expected_hash
        except (OSError, URLError):
            if attempt == 2:
                raise
            time.sleep(attempt + 1)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    raise RuntimeError("Unreachable download state")


def download_dataset(raw_dir: Path, records: list[str]) -> dict[str, str]:
    records = validate_records(records)
    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    # The checksum listing is the trust source fetched from the official HTTPS site.
    with urlopen(BASE_URL + "SHA256SUMS.txt", timeout=60) as response:
        checksum_bytes = response.read()
    checksums = {}
    for line in checksum_bytes.decode().splitlines():
        digest, filename = line.split(maxsplit=1)
        checksums[filename.lstrip("*").removeprefix("./")] = digest
    (raw_dir / "SHA256SUMS.txt").write_bytes(checksum_bytes)
    filenames = list(METADATA) + [
        f"{record}.{extension}" for record in records for extension in ("hea", "dat", "apn", "qrs")
    ]
    hashes = {}
    for name in filenames:
        if name not in checksums:
            raise ValueError(f"No published checksum for {name}")
        hashes[name] = fetch_verified(BASE_URL + name, raw_dir / name, checksums[name])
        print(f"Verified {name}", flush=True)
    # Preserve provenance for every verified cached record, including earlier invocations.
    existing_path = raw_dir / "download-manifest.json"
    existing = json.loads(existing_path.read_text()) if existing_path.exists() else {}
    existing_hashes = existing.get("sha256", {})
    existing_hashes.update(hashes)
    manifest = {
        "dataset": "apnea-ecg",
        "version": "1.0.0",
        "source_url": BASE_URL,
        "checksum_listing_sha256": hashlib.sha256(checksum_bytes).hexdigest(),
        "sha256": existing_hashes,
    }
    temporary = existing_path.with_suffix(".json.part")
    temporary.write_text(json.dumps(manifest, indent=2) + "\n")
    temporary.replace(existing_path)
    return hashes

## Implementation · Configuration validation

The following code is the complete implementation for this stage. Continue to the numbered workflow below to run it.

In [3]:
def load_config(source: Path | dict) -> dict:
    config = tomllib.loads(source.read_text()) if isinstance(source, Path) else source.copy()
    numeric = [
        "fs",
        "window_seconds",
        "filter_order",
        "low_hz",
        "high_hz",
        "rr_min_seconds",
        "rr_max_seconds",
        "min_usable_rr",
        "flat_std_mv",
        "low_amplitude_span_mv",
        "seed",
    ]
    try:
        if any(not math.isfinite(config[name]) for name in numeric):
            raise ValueError("Settings must be finite")
        if config["fs"] != 100 or config["window_seconds"] != 60:
            raise ValueError("This study requires 100 Hz ECG and 60-second windows")
        if not 0 < config["low_hz"] < config["high_hz"] < 50:
            raise ValueError("Filter cutoffs must satisfy 0 < low < high < 50 Hz")
        if not 0 < config["rr_min_seconds"] < config["rr_max_seconds"]:
            raise ValueError("RR range must be positive and increasing")
        for name, minimum in [
            ("fs", 1),
            ("window_seconds", 1),
            ("filter_order", 1),
            ("min_usable_rr", 1),
            ("seed", 0),
        ]:
            if type(config[name]) is not int or config[name] < minimum:
                raise ValueError(f"Invalid {name}")
        if config["flat_std_mv"] <= 0 or config["low_amplitude_span_mv"] <= 0:
            raise ValueError("Signal amplitude thresholds must be positive")
        if set(config["exclude_records"]) - set(LEARNING_RECORDS):
            raise ValueError("Unknown excluded record")
        groups = config["groups"]
        if set(groups) - set(LEARNING_RECORDS) or any(
            not isinstance(group, str) or not group.strip() for group in groups.values()
        ):
            raise ValueError("Invalid source-group mapping")
        if groups.get("c05", "c05") != groups.get("c06", "c06"):
            raise ValueError("Known overlapping c05/c06 records must share a source group")
    except (KeyError, TypeError) as exc:
        raise ValueError(f"Missing or invalid study configuration: {exc}") from exc
    return config

## Implementation · ECG filtering, beat detection, RR and windows

The following code is the complete implementation for this stage. Continue to the numbered workflow below to run it.

In [4]:
WINDOW_COLUMNS = [
    "record_id",
    "minute",
    "start_sample",
    "end_sample",
    "label",
    "n_peaks",
    "n_rr",
    "n_usable_rr",
    "suspect_rr",
    "too_few_rr",
]
REJECT_COLUMNS = ["record_id", "start_sample", "symbol", "reason"]


def filter_ecg(signal, fs: int, config: dict) -> np.ndarray:
    signal = np.asarray(signal, dtype=float)
    if signal.ndim != 1 or not np.isfinite(signal).all():
        raise ValueError("ECG must be a finite one-dimensional signal")
    if fs != config["fs"]:
        raise ValueError(f"Expected {config['fs']} Hz, got {fs}")
    sos = butter(
        config["filter_order"],
        [config["low_hz"], config["high_hz"]],
        btype="bandpass",
        fs=fs,
        output="sos",
    )
    # SciPy's default odd extension is used at recording edges, never across records.
    return sosfiltfilt(sos, signal)


def detect_r_peaks(signal, fs: int) -> np.ndarray:
    signal = np.asarray(signal, dtype=float)
    if signal.ndim != 1 or not np.isfinite(signal).all():
        raise ValueError("ECG must be finite and one-dimensional")
    if np.ptp(signal) == 0:
        return np.array([], dtype=np.int64)
    # Bound adaptive-threshold drift/backsearch after artifacts. These are
    # detection blocks, not labeled minute windows: 10 s overlap preserves
    # context around each core block, and each refined peak has one owner.
    block, overlap = 300 * fs, 10 * fs
    detected = []
    for core_start in range(0, len(signal), block):
        core_end = min(len(signal), core_start + block)
        left, right = max(0, core_start - overlap), min(len(signal), core_end + overlap)
        segment = signal[left:right]
        if np.ptp(segment) == 0:
            continue
        # GQRS distinguishes narrow QRS complexes from prominent broad T waves
        # that XQRS double-counted in the 100 Hz a02/b02 morphology review.
        low, median, high = np.quantile(segment, [0.005, 0.5, 0.995])
        span = high - low
        if span < 1e-6:
            continue
        # GQRS has amplitude-dependent thresholds. Normalize detection input
        # within this block only; saved waveforms retain their physical mV.
        normalized = (segment - median) / span
        peaks = np.asarray(gqrs_detect(sig=normalized, fs=fs), dtype=np.int64)
        if len(peaks):
            peaks = (
                correct_peaks(
                    segment,
                    peaks,
                    search_radius=round(0.1 * fs),
                    smooth_window_size=max(3, round(0.05 * fs)),
                    peak_dir="both",
                )
                + left
            )
            detected.extend(peaks[(peaks >= core_start) & (peaks < core_end)])
    return np.unique(np.asarray(detected, dtype=np.int64))


def rr_table(record_id: str, peaks, fs: int, config: dict) -> pd.DataFrame:
    peaks = np.asarray(peaks)
    if peaks.ndim != 1 or not np.issubdtype(peaks.dtype, np.integer):
        raise ValueError("R peak indices must be an integer vector")
    if (peaks < 0).any() or (np.diff(peaks) <= 0).any():
        raise ValueError("R peak indices must be positive and strictly increasing")
    rr = np.diff(peaks) / fs
    return pd.DataFrame(
        {
            "record_id": [record_id] * len(rr),
            "start_sample": peaks[:-1],
            "end_sample": peaks[1:],
            "rr_seconds": rr,
            "plausible": (rr >= config["rr_min_seconds"]) & (rr <= config["rr_max_seconds"]),
        }
    )


def make_windows(record_id, n_samples, annotation_samples, symbols, peaks, config):
    fs = config["fs"]
    width = fs * config["window_seconds"]
    positions = np.asarray(annotation_samples)
    if len(positions) != len(symbols) or positions.ndim != 1:
        raise ValueError("Annotation positions and symbols must match")
    if len(positions) and (
        not np.issubdtype(positions.dtype, np.integer)
        or (positions < 0).any()
        or (positions % width != 0).any()
        or (np.diff(positions) <= 0).any()
    ):
        raise ValueError("Annotations must be unique, ordered, nonnegative minute boundaries")
    if set(symbols) - {"A", "N"}:
        raise ValueError("Unexpected apnea annotation symbol")
    peaks = np.asarray(peaks)
    rr = rr_table(record_id, peaks, fs, config)
    if len(peaks) and peaks[-1] >= n_samples:
        raise ValueError("R peak outside signal bounds")
    rows, rejected = [], []
    mapping = dict(zip(positions, symbols, strict=True))
    for start in range(0, n_samples, width):
        symbol = mapping.get(start)
        if symbol is None or start + width > n_samples:
            rejected.append(
                {
                    "record_id": record_id,
                    "start_sample": start,
                    "symbol": symbol,
                    "reason": "missing_annotation" if symbol is None else "incomplete_window",
                }
            )
            continue
        end = start + width
        first = np.searchsorted(peaks, start)
        last = np.searchsorted(peaks, end)
        # Both RR endpoints must be in this minute: crossing intervals are retained
        # in the record RR table but do not enter minute-level HRV statistics.
        local_rr = rr.iloc[first : max(first, last - 1)]
        usable = int(local_rr.plausible.sum())
        rows.append(
            {
                "record_id": record_id,
                "minute": start // width,
                "start_sample": start,
                "end_sample": end,
                "label": int(symbol == "A"),
                "n_peaks": int(last - first),
                "n_rr": len(local_rr),
                "n_usable_rr": usable,
                "suspect_rr": bool((~local_rr.plausible).any()),
                "too_few_rr": usable < config["min_usable_rr"],
            }
        )
    for start, symbol in mapping.items():
        if start >= n_samples:
            rejected.append(
                {
                    "record_id": record_id,
                    "start_sample": start,
                    "symbol": symbol,
                    "reason": "annotation_outside_signal",
                }
            )
    return pd.DataFrame(rows, columns=WINDOW_COLUMNS), pd.DataFrame(
        rejected, columns=REJECT_COLUMNS
    )

## Historical implementation · Recording-group partitions

This function is retained to reproduce the historical recording-disjoint assignments. New runs leave partitions unassigned until Notebook 01 freezes the grouped split.

In [5]:
def assign_splits(record_labels: dict, groups: dict, seed: int) -> pd.DataFrame:
    records = sorted(record_labels)
    group_ids = np.array([groups.get(record, record) for record in records])
    if len(set(group_ids)) < 5:
        raise ValueError("At least five independent groups are required for a 60/20/20 split")
    if any(not labels or set(labels) - {0, 1} for labels in record_labels.values()):
        raise ValueError("Every record needs nonempty binary class labels")
    # Choose the first seeded group assignment that gives both classes in every
    # partition. This constraint uses labels, never model performance or quality.
    for offset in range(1000):
        train_val, test = next(
            GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed + offset).split(
                records, groups=group_ids
            )
        )
        train_rel, val_rel = next(
            GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed + offset).split(
                train_val, groups=group_ids[train_val]
            )
        )
        partitions = {"train": train_val[train_rel], "validation": train_val[val_rel], "test": test}
        if all(
            {label for i in indices for label in record_labels[records[i]]} == {0, 1}
            for indices in partitions.values()
        ):
            rows = [
                {"record_id": records[i], "group_id": group_ids[i], "split": name}
                for name, indices in partitions.items()
                for i in indices
            ]
            table = pd.DataFrame(rows).sort_values("record_id").reset_index(drop=True)
            if table.groupby("group_id").split.nunique().max() != 1:
                raise ValueError("A source group crosses partitions")
            table.attrs["effective_seed"] = seed + offset
            return table
    raise ValueError("Cannot establish both classes in each partition with these source groups")

## Implementation · Inspection figures

The following code is the complete implementation for this stage. Continue to the numbered workflow below to run it.

In [6]:
matplotlib.use("Agg")


def create_figures(output: Path, windows: pd.DataFrame, rr: pd.DataFrame, fs: int) -> None:
    folder = output / "figures"
    folder.mkdir()
    for label, symbol in [(0, "N"), (1, "A")]:
        candidates = windows[windows.label == label]
        if candidates.empty:
            continue
        row = candidates.iloc[0]
        with np.load(output / "waveforms" / f"{row.record_id}.npz") as wave:
            start, end = int(row.start_sample), int(row.end_sample)
            raw = wave["raw"][start:end]
            filtered = wave["filtered"][start:end]
        peaks = pd.read_parquet(output / "peaks.parquet")
        peaks = peaks[
            (peaks.record_id == row.record_id)
            & (peaks["sample"] >= start)
            & (peaks["sample"] < end)
        ]["sample"].to_numpy()
        local = peaks - start
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
        for axis, stop in zip(axes, [len(raw), min(1000, len(raw))], strict=True):
            axis.plot(np.arange(stop) / fs, raw[:stop], label="Raw", alpha=0.6, lw=0.8)
            axis.plot(np.arange(stop) / fs, filtered[:stop], label="Filtered", lw=0.8)
            selected = local[local < stop]
            axis.scatter(selected / fs, filtered[selected], c="tab:red", s=14, label="R peaks")
            axis.set(xlabel="Seconds within minute", ylabel="ECG (mV)")
            axis.legend(loc="upper right")
        fig.suptitle(f"{row.record_id}, minute {row.minute}: {symbol} annotation (illustrative)")
        fig.savefig(folder / f"example_{symbol}.png", dpi=150)
        plt.close(fig)

    fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
    windows.label.value_counts().reindex([0, 1], fill_value=0).plot.bar(ax=axes[0])
    axes[0].set(title="Labeled windows", xlabel="0 = N; 1 = A", ylabel="Count")
    rr.rr_seconds.hist(bins=80, range=(0, 3), ax=axes[1])
    axes[1].set(title="RR intervals (display range 0–3 s)", xlabel="Seconds", ylabel="Count")
    windows.groupby(["split", "label"]).size().unstack(fill_value=0).plot.bar(ax=axes[2])
    axes[2].set(title="Partition class counts", ylabel="Windows")
    fig.savefig(folder / "overview.png", dpi=150)
    plt.close(fig)

## Implementation · Preparation and saved artifacts

The following code is the complete implementation for this stage. Continue to the numbered workflow below to run it.

In [7]:
def prepare(
    raw_dir: Path, output_dir: Path, records: list[str], config: dict, make_split: bool = False
) -> dict:
    """Write all artifacts atomically to a new directory; never mix partial/full runs."""
    raw_dir, output_dir = Path(raw_dir), Path(output_dir)
    config = load_config(config)
    records = validate_records(records)
    if output_dir.exists():
        raise FileExistsError(f"Choose a new output directory; {output_dir} already exists")
    if make_split and set(records) != set(LEARNING_RECORDS):
        raise ValueError("Evaluation splitting requires all 35 learning records; use --no-split")
    retained = [r for r in records if r not in config["exclude_records"]]
    if not retained:
        raise ValueError("No records remain after exclusions")
    checksums = {}
    for line in (raw_dir / "SHA256SUMS.txt").read_text().splitlines():
        digest, name = line.split(maxsplit=1)
        checksums[name.lstrip("*").removeprefix("./")] = digest
    source_hashes = {}
    for record in retained:
        for extension in ("hea", "dat", "apn"):
            name = f"{record}.{extension}"
            digest = sha256(raw_dir / name)
            if digest != checksums.get(name):
                raise ValueError(f"Source checksum mismatch: {name}; rerun download")
            source_hashes[name] = digest
    output_dir.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix=".phase1-", dir=output_dir.parent) as temporary:
        stage = Path(temporary) / "run"
        stage.mkdir()
        (stage / "waveforms").mkdir()
        all_windows, all_rr, all_peaks, all_rejected, quality = [], [], [], [], []
        for record_id in retained:
            print(f"Processing {record_id}", flush=True)
            path = str(raw_dir / record_id)
            record = wfdb.rdrecord(path)
            if record.n_sig != 1 or record.fs != config["fs"] or record.units != ["mV"]:
                raise ValueError(
                    f"{record_id}: expected one ECG channel in mV at {config['fs']} Hz"
                )
            raw = record.p_signal[:, 0]
            if len(raw) != record.sig_len:
                raise ValueError(f"{record_id}: header/signal length mismatch")
            filtered = filter_ecg(raw, record.fs, config)
            peaks = detect_r_peaks(filtered, record.fs)
            annotations = wfdb.rdann(path, "apn")
            windows, rejected = make_windows(
                record_id, len(raw), annotations.sample, annotations.symbol, peaks, config
            )
            if windows.empty:
                raise ValueError(f"{record_id}: no complete labeled windows")
            windows["flat_signal"] = [
                bool(
                    np.std(raw[int(row.start_sample) : int(row.end_sample)]) < config["flat_std_mv"]
                )
                for row in windows.itertuples()
            ]
            windows["amplitude_span_mv"] = [
                float(
                    np.diff(
                        np.quantile(
                            raw[int(row.start_sample) : int(row.end_sample)], [0.005, 0.995]
                        )
                    )[0]
                )
                for row in windows.itertuples()
            ]
            windows["low_amplitude"] = windows.amplitude_span_mv < config["low_amplitude_span_mv"]
            windows["quality_flag"] = (
                windows.flat_signal
                | windows.too_few_rr
                | windows.suspect_rr
                | windows.low_amplitude
            )
            windows["filter_edge"] = (windows.start_sample < config["fs"] * 5) | (
                windows.end_sample > len(raw) - config["fs"] * 5
            )
            np.savez_compressed(
                stage / "waveforms" / f"{record_id}.npz",
                raw=raw,
                filtered=filtered,
                fs=np.array(record.fs),
            )
            all_windows.append(windows)
            intervals = rr_table(record_id, peaks, record.fs, config)
            all_rr.append(intervals)
            all_peaks.append(pd.DataFrame({"record_id": record_id, "sample": peaks}))
            all_rejected.append(rejected)
            quality.append(
                {
                    "record_id": record_id,
                    "n_samples": len(raw),
                    "n_annotations": len(annotations.sample),
                    "n_windows": len(windows),
                    "n_apnea": int(windows.label.sum()),
                    "n_non_apnea": int((windows.label == 0).sum()),
                    "n_peaks": len(peaks),
                    "n_suspect_rr": int((~intervals.plausible).sum()),
                    "n_flagged_windows": int(windows.quality_flag.sum()),
                    "n_rejected_windows": len(rejected),
                    "finite_signal": True,
                }
            )
        windows = pd.concat(all_windows, ignore_index=True)
        rr = pd.concat(all_rr, ignore_index=True)
        labels = windows.groupby("record_id").label.apply(list).to_dict()
        if make_split:
            splits = assign_splits(labels, config["groups"], config["seed"])
        else:
            splits = pd.DataFrame(
                {
                    "record_id": retained,
                    "group_id": [config["groups"].get(r, r) for r in retained],
                    "split": "unassigned",
                }
            )
        windows = windows.merge(splits, on="record_id", validate="many_to_one")
        windows.to_parquet(stage / "windows.parquet", index=False)
        rr.to_parquet(stage / "rr.parquet", index=False)
        pd.concat(all_peaks, ignore_index=True).to_parquet(stage / "peaks.parquet", index=False)
        rejected = pd.concat(all_rejected, ignore_index=True)
        rejected.to_parquet(stage / "rejected_windows.parquet", index=False)
        pd.DataFrame(quality).to_parquet(stage / "record_quality.parquet", index=False)
        splits.to_csv(stage / "splits.csv", index=False)
        counts = windows.groupby(["split", "label"]).size().rename("n_windows").reset_index()
        counts.to_csv(stage / "class_counts.csv", index=False)
        create_figures(stage, windows, rr, config["fs"])
        source_dir = ROOT / "notebooks"
        revision = subprocess.run(
            ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=False
        ).stdout.strip()
        manifest = {
            "dataset": "apnea-ecg",
            "dataset_version": "1.0.0",
            "source_url": BASE_URL,
            "created_utc": datetime.now(UTC).isoformat(),
            "config": config,
            "requested_records": records,
            "processed_records": retained,
            "excluded_records": sorted(set(records) - set(retained)),
            "n_windows": len(windows),
            "n_apnea": int(windows.label.sum()),
            "n_non_apnea": int((windows.label == 0).sum()),
            "n_flagged_windows": int(windows.quality_flag.sum()),
            "n_rejected_windows": len(rejected),
            "class_counts": counts.to_dict("records"),
            "split_status": (
                "development_holdout"
                if make_split
                else "unsplit_prepared"
                if set(records) == set(LEARNING_RECORDS)
                else "smoke_only"
            ),
            "effective_split_seed": splits.attrs.get("effective_seed"),
            "independence": (
                "recording-disjoint; patient identities unverified"
                if make_split
                else "unassigned; patient identities unverified"
            ),
            "label_semantics": (
                "A/N status at beginning of associated minute; apnea includes hypopnea"
            ),
            "rr_window_policy": "both endpoints inside half-open minute; no RR repair",
            "filter": "record-level zero-phase SOS; default odd padding; 5 s edge windows flagged",
            "detector": (
                "WFDB GQRS (default parameters): 300 s blocks, 10 s margins; "
                "detector-only median centering and 0.5–99.5 percentile span normalization; "
                "correct_peaks radius 0.1 s, smoothing 0.05 s, both polarities"
            ),
            "quality_policy": (
                "retain complete finite windows with flags; no quality-based exclusions"
            ),
            "python": platform.python_version(),
            "git_revision": revision,
            "packages": {
                name: version(name)
                for name in [
                    "numpy",
                    "scipy",
                    "pandas",
                    "pyarrow",
                    "matplotlib",
                    "wfdb",
                    "scikit-learn",
                ]
            },
            "source_sha256": source_hashes,
            "code_sha256": {p.name: sha256(p) for p in sorted(source_dir.glob("*.ipynb"))},
            "artifact_sha256": {
                str(p.relative_to(stage)): sha256(p)
                for p in sorted(stage.rglob("*"))
                if p.is_file()
            },
            "limitations": [
                "Patient IDs unavailable; recording groups do not prove subject independence.",
                "RR intervals are detected beat intervals, not validated NN intervals.",
                "Offline zero-phase filtering uses future samples within each recording.",
                "x evaluation records/labels not used; external evaluation remains separate.",
            ],
        }
        (stage / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
        (stage / "report.md").write_text(
            "# Phase 1 preparation report\n\n"
            f"Records: {len(retained)}; windows: {len(windows)}; "
            f"apnea: {manifest['n_apnea']}; non-apnea: {manifest['n_non_apnea']}.\n\n"
            f"Flagged (retained): {manifest['n_flagged_windows']}; "
            f"rejected/missing: {len(rejected)}.\n\n"
            f"Split status: {manifest['split_status']}.\n\n"
            + "\n".join(f"- {item}" for item in manifest["limitations"])
            + "\n"
        )
        stage.rename(output_dir)
    return manifest

## 1. Freeze the preparation settings

The config specifies 100 Hz, 60-second windows, filtering and RR plausibility limits, quality flags, seed 67, and the known `c05`/`c06` source overlap. `c05` is excluded. Other patient identities have not been verified.

In [8]:
config = load_config(CONFIG)
display(pd.Series({k: v for k, v in config.items() if k != "groups"}, name="value").to_frame())
print("Verified source groups:", config["groups"])

,value
fs,100
window_seconds,60
filter_order,4
low_hz,0.5
high_hz,40.0
rr_min_seconds,0.3
rr_max_seconds,2.0
min_usable_rr,20
flat_std_mv,0.001
low_amplitude_span_mv,0.1


Verified source groups: {'c05': 'c05_c06', 'c06': 'c05_c06'}


## 2. Download and verify source files

If the prepared run is absent, the download function retrieves the learning records and checks PhysioNet SHA-256 sums. It checks cached files too. Preparation independently checks the source checksums again. The first run requires network access and disk space.

In [9]:
if RUN.exists():
    print("Reusing prepared run:", RUN)
else:
    download_dataset(RAW, LEARNING_RECORDS)
    print("Downloaded and verified", len(LEARNING_RECORDS), "learning records")

Verified RECORDS


Verified ANNOTATORS


Verified additional-information.txt


Verified a01.hea


Verified a01.dat


Verified a01.apn


... (135 outputs left out)

Verified c10.apn


Verified c10.qrs


Downloaded and verified

 35 learning records


## 3. Prepare complete labeled minutes

`prepare` filters each recording, detects R peaks, creates RR intervals and annotation-aligned minutes, applies descriptive quality flags, and saves the prepared cohort with unassigned partitions. The historical Phase 1 split remains in existing runs; new runs receive the grouped split after inspection in Notebook 01. It stages the run and publishes only after successful completion. Existing output directories are never overwritten.

In [10]:
if RUN.exists():
    manifest = json.loads((RUN / "manifest.json").read_text())
else:
    manifest = prepare(RAW, RUN, LEARNING_RECORDS, config, make_split=False)
print("Records:", len(manifest["processed_records"]), "Windows:", manifest["n_windows"])
print("Split status:", manifest["split_status"])
print("Independence:", manifest["independence"])

Processing a01


Processing a02


Processing a03


Processing a04


Processing a05


Processing a06


... (25 outputs left out)

Processing c08


Processing c09


Processing c10


Records: 34 Windows: 16558
Split status: unsplit_prepared
Independence: unassigned; patient identities unverified


## 4. Inspect labels and quality

Every accepted minute is 6,000 samples long. Phase 1 retains flagged complete windows for an explicit downstream policy. The tables below make cohort composition and signal-quality concerns visible before modeling.

In [11]:
windows = pd.read_parquet(RUN / "windows.parquet")
quality = pd.read_parquet(RUN / "record_quality.parquet")
rejected = pd.read_parquet(RUN / "rejected_windows.parquet")
assert (windows.end_sample - windows.start_sample).eq(6000).all()
assert windows.groupby("group_id").split.nunique().max() == 1
assert windows.split.isin(["train", "validation", "test", "unassigned"]).all()
display(windows.groupby("label").size().rename("windows").to_frame())
display(
    windows[
        ["quality_flag", "suspect_rr", "too_few_rr", "flat_signal", "low_amplitude", "filter_edge"]
    ]
    .sum()
    .rename("flagged minutes")
    .to_frame()
)
display(rejected.groupby("reason").size().rename("rejected windows").to_frame())

,windows
label,
0,10050
1,6508


,flagged minutes
quality_flag,623
suspect_rr,552
too_few_rr,115
flat_signal,17
low_amplitude,52
filter_edge,37


,rejected windows
reason,
incomplete_window,21
missing_annotation,52


## 5. View the ECG and detected beats

The source waveform stays in physical mV; the saved filtered waveform is for retrospective analysis. Choose another record/minute below to inspect a concerning segment. The automated peaks are not audited normal-to-normal beats.

In [12]:
record_id, minute = "a01", 0
row = windows[(windows.record_id == record_id) & (windows.minute == minute)].iloc[0]
with np.load(RUN / "waveforms" / f"{record_id}.npz") as wave:
    raw = wave["raw"][int(row.start_sample) : int(row.end_sample)]
    filtered = wave["filtered"][int(row.start_sample) : int(row.end_sample)]
peaks = pd.read_parquet(RUN / "peaks.parquet")
local = (
    peaks.loc[
        (peaks.record_id == record_id)
        & peaks["sample"].between(row.start_sample, row.end_sample - 1),
        "sample",
    ].to_numpy()
    - row.start_sample
)
fig, ax = plt.subplots(figsize=(12, 3))
t = np.arange(1000) / 100
ax.plot(t, raw[:1000], alpha=0.5, label="raw")
ax.plot(t, filtered[:1000], label="filtered")
near = local[local < 1000]
ax.scatter(near / 100, filtered[near], s=16, label="R peaks")
ax.set(
    xlabel="Seconds within minute",
    ylabel="ECG (mV)",
    title=f"{record_id}, minute {minute}, label {row.label}",
)
ax.legend()
display(fig)
plt.close(fig)

<Figure size 1200x300 with 1 Axes>

## Continue

Use [01_data_inspection.ipynb](01_data_inspection.ipynb) for deeper quality review, then [02_feature_ablation.ipynb](02_feature_ablation.ipynb) for Phase 2. The partitions here are recording-disjoint, **not confirmed patient-independent**.